In [4]:
import pandas as pd
import numpy as np
import timeit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import pearsonr, spearmanr

df_power = pd.read_csv('household_power_consumption.txt', sep=';', na_values=['?'], low_memory=False)
df_power = df_power.dropna()
df_power['Date'] = pd.to_datetime(df_power['Date'], format='%d/%m/%Y')
df_power['Time'] = pd.to_datetime(df_power['Time'], format='%H:%M:%S').dt.time
display(df_power.head())

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [5]:
def query_1(df):
    return df[df['Global_active_power'] > 5.0]

def query_2(df):
    mask1 = df['Global_intensity'].between(19, 20)
    mask2 = (df['Sub_metering_1'] + df['Sub_metering_2']) > df['Sub_metering_3']
    return df[mask1 & mask2]

def query_3(df):
    sample = df.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def query_4(df):
    df_filtered = df[(df['Time'] >= pd.to_datetime('18:00:00').time()) & (df['Global_active_power'] > 6)]
    mask = (df_filtered['Sub_metering_2'] > df_filtered['Sub_metering_1']) & \
           (df_filtered['Sub_metering_2'] > df_filtered['Sub_metering_3'])
    df_result = df_filtered[mask].reset_index(drop=True)
    half = len(df_result) // 2
    first_half = df_result.iloc[:half].iloc[::3]
    second_half = df_result.iloc[half:].iloc[::4]
    return pd.concat([first_half, second_half])

print("Час запиту 1:", timeit.timeit(lambda: query_1(df_power), number=10))
print("Час запиту 2:", timeit.timeit(lambda: query_2(df_power), number=10))
print("Час запиту 3:", timeit.timeit(lambda: query_3(df_power), number=10))
print("Час запиту 4:", timeit.timeit(lambda: query_4(df_power), number=10))

Час запиту 1: 0.06642370000190567
Час запиту 2: 0.18854629999987083
Час запиту 3: 2.60055740000098
Час запиту 4: 0.8957481000034022


In [6]:
sample_df = df_power[['Global_active_power', 'Global_intensity']].head(1000).copy()

scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

sample_df[['norm_power', 'norm_intensity']] = scaler_minmax.fit_transform(sample_df[['Global_active_power', 'Global_intensity']])
sample_df[['std_power', 'std_intensity']] = scaler_std.fit_transform(sample_df[['Global_active_power', 'Global_intensity']])

pearson_corr, _ = pearsonr(sample_df['Global_active_power'], sample_df['Global_intensity'])
spearman_corr, _ = spearmanr(sample_df['Global_active_power'], sample_df['Global_intensity'])

print(f"Пірсон: {pearson_corr:.4f}")
print(f"Спірмен: {spearman_corr:.4f}")

df_power['Time_of_day'] = pd.cut(pd.to_datetime(df_power['Time'], format='%H:%M:%S').dt.hour, 
                                 bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)
df_ohe = pd.get_dummies(df_power['Time_of_day'].head(100), prefix='Time')
display(df_ohe.head())

Пірсон: 0.9955
Спірмен: 0.9927


,Time_Night,Time_Morning,Time_Afternoon,Time_Evening
0,False,False,True,False
1,False,False,True,False
2,False,False,True,False
3,False,False,True,False
4,False,False,True,False
